# CA2 - Data Preparation and Exploratory Data Analysis

**Project:** Predicting Orbital Decay of Low Earth Orbit Objects to Support Satellite Operators' Disposal Planning

**Student:** Oleksii Taranov (sba26106)

**Module:** Strategic Thinking

**GitHub:** https://github.com/sba26106-bit/strategic-think-ca1

## 1. Business understanding
Satellite operators must show that hardware left in low Earth orbit (LEO) will re-enter within **five years** after the mission ends (FCC, 2024; ESA, 2023). The final model will estimate the **remaining orbital lifetime** of an uncontrolled object from its current orbit and solar activity.

## 2. Data understanding

In [1]:
# Import libraries

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pyarrow.parquet as pq
from scipy import stats

In [2]:
# Folders and settings used in the whole notebook

DATA = Path("data")                      # the data folder path
RAW = DATA / "raw"                       # original downloads - this notebook never changes them
TLE_DIR = RAW / "tle_history"            # raw orbit history: 68 yearly Parquet files
OUT = DATA / "processed" / "ca2"         # everything this notebook writes goes here
FIG = Path("figures")                    # saved charts for the report

SATCAT_FILE = RAW / "celestrak_satcat_2026-10-04.csv"
SPACEWEATHER_FILE = RAW / "spaceweather_2026-10-04.csv"
CATALOG_DATE = pd.Timestamp("2026-10-04")   # SATCAT snapshot date = end of observation (censoring date)

R_EARTH = 6378.137        # Earth radius (km)
MU = 398600.4418          # Earth's gravitational parameter (km^3/s^2)
LEO_MAX_KM = 2000         # LEO upper limit: apogee must be at or below this height
MIN_PERIGEE_KM = 80       # records with a lower perigee are bad orbit fits
SEED = 42                 # fixed random seed, so samples and splits can be reproduced

sns.set_theme(style="whitegrid")

print(f"Data folder: {DATA.resolve()}")
print(f"Raw orbit history files: {len(list(TLE_DIR.glob('tle_*.parquet')))}")
print(f"SATCAT found: {SATCAT_FILE.exists()} | Space weather found: {SPACEWEATHER_FILE.exists()}")

Data folder: C:\Users\oleks\Documents\CCT College Dublin\Strategical thinking\CA2\Submission\data
Raw orbit history files: 68
SATCAT found: True | Space weather found: True


### 2.1 Sources

The sources used in this project are listed below. The amounts are the **expected (approximate)** sizes, taken from the source descriptions and the download step. The exact numbers of rows and objects are calculated with Python in the next section.

| Source | File | Content | Expected amount | Permission | Used in |
|---|---|---|---|---|---|
| Space-Track.org orbit history, via a public Parquet copy (Simon, 2026) | data/raw/tle_history/ | Orbital elements, drag term and decay rate for every tracked object, 1959-2026 | About 239.5 million records in 68 yearly files (about 12.5 GB) | Basic data redistributable with citation (Space-Track.org, 2026); copy under CC BY 4.0 licence | CA2 |
| CelesTrak SATCAT (CelesTrak, 2026a) | data/raw/celestrak_satcat_2026-10-04.csv | Object type, status, size class, launch and re-entry dates | About 70,000 catalogued objects since 1957 | Free public data; the CelesTrak usage policy is followed; CelesTrak and the original providers are cited | CA2 |
| CelesTrak space weather (CelesTrak, 2026b) | data/raw/spaceweather_2026-10-04.csv | Daily F10.7 solar flux, Kp/Ap indices, sunspot number | About 25,000 daily rows since October 1957 | Free public data; CelesTrak and the original providers are cited | CA2 |
| NOAA solar-cycle forecast (NOAA Space Weather Prediction Center, 2026) | data/raw/noaa_predicted_solar_cycle_2026-10-04.json | Predicted monthly F10.7 and sunspot number | Monthly values for the current solar cycle | US government work (public domain);<br>NOAA Space Weather Prediction Center is cited | Semester 2 |
| ESA DISCOSweb (European Space Agency, 2026) | data/raw/discos_objects/ | Mass, cross-section and initial orbits of catalogued objects, including fragmentation debris | About 93,000 objects | Registered account;<br>ESA DISCOSweb is cited;<br>raw data is not republished | Semester 2 |
| GCAT (McDowell, 2020) | data/raw/gcat_satcat_2026-10-04.tsv | Independent mass and size estimates for cross-checking;<br>payload end-of-life data | About 70,000 catalogued objects | Creative Commons CC-BY licence;<br>GCAT is cited | Semester 2 |

The NOAA forecast, DISCOSweb and GCAT are not used in this phase. They are planned for Semester 2 (solar-activity forecast, mass and size, payload end-of-life dates).